# Zadanie 2 — Neo4j: wspólne relacje

## Struktura grafu

**Węzły:**
- `User` — użytkownik
- `Device` — urządzenie

**Relacja:**
```
(:User)-[:USES]->(:Device)
```

## Dane — tworzenie grafu

In [ ]:
# Instalacja sterownika Neo4j (jeśli potrzebna)
# !pip install neo4j

In [1]:
from neo4j import GraphDatabase
import pandas as pd

# Połączenie z Neo4j
URI = "bolt://neo4j_nosql_lab:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"  

In [2]:
def load_sample_data(tx):
    tx.run("""
       // FILMY
       CREATE (matrix:Movie {title: 'The Matrix', released: 1999}),
       (johnwick:Movie {title: 'John Wick', released: 2014}),
       (inception:Movie {title: 'Inception', released: 2010}),
       (tenet:Movie {title: 'Tenet', released: 2020}),

       // OSOBY
       (keanu:Person {name: 'Keanu Reeves'}),
       (laurence:Person {name: 'Laurence Fishburne'}),
       (lana:Person {name: 'Lana Wachowski'}),
       (chad:Person {name: 'Chad Stahelski'}),
       (nolan:Person {name: 'Christopher Nolan'}),
       (leo:Person {name: 'Leonardo DiCaprio'}),
       (elizabeth:Person {name: 'Elizabeth Debicki'}),

       // GATUNKI
       (scifi:Genre {name: 'Sci-Fi'}),
       (action:Genre {name: 'Action'}),
       (thriller:Genre {name: 'Thriller'}),

       // OCENY
       (user1:User {name: 'student1'}),
       (user2:User {name: 'student2'}),

       // RELACJE FILM - OSOBY
       (keanu)-[:ACTED_IN]->(matrix),
       (laurence)-[:ACTED_IN]->(matrix),
       (lana)-[:DIRECTED]->(matrix),

       (keanu)-[:ACTED_IN]->(johnwick),
       (chad)-[:DIRECTED]->(johnwick),

       (leo)-[:ACTED_IN]->(inception),
       (nolan)-[:DIRECTED]->(inception),

       (elizabeth)-[:ACTED_IN]->(tenet),
       (nolan)-[:DIRECTED]->(tenet),

       // RELACJE FILM - GATUNEK
       (matrix)-[:HAS_GENRE]->(scifi),
       (matrix)-[:HAS_GENRE]->(action),
       (johnwick)-[:HAS_GENRE]->(action),
       (inception)-[:HAS_GENRE]->(scifi),
       (inception)-[:HAS_GENRE]->(thriller),
       (tenet)-[:HAS_GENRE]->(scifi),
       (tenet)-[:HAS_GENRE]->(action),

       // RELACJE SPOŁECZNE
       (keanu)-[:FOLLOWS]->(laurence),
       (lana)-[:FOLLOWS]->(nolan),
       (leo)-[:FOLLOWS]->(keanu),

       // RELACJE OCENIANIA
       (user1)-[:RATED {score: 5}]->(matrix),
       (user1)-[:RATED {score: 4}]->(johnwick),
       (user2)-[:RATED {score: 3}]->(tenet),
       (user2)-[:RATED {score: 4}]->(inception)
    """)

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    session.execute_write(load_sample_data)

print("Dane zostały załadowane!")

driver.close()


Dane zostały załadowane!


In [8]:
def run_cypher(driver, query: str):
    with driver.session() as session:
        result = session.run(query)
        records = list(result)
        
        if not records:
            print("Brak wyników.")
            return
        
        df = pd.DataFrame([r.data() for r in records])
        return df


In [6]:
def qexec(query: str):
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    
    try:
        df = run_cypher(driver, query)
    finally:
        driver.close()
    
    return df

In [10]:
# Wyczyszczenie bazy i załadowanie danych
qexec("MATCH (n) DETACH DELETE n")

CREATE_DATA = """
CREATE
  (u1:User {name: 'U1'}),
  (u2:User {name: 'U2'}),
  (u3:User {name: 'U3'}),
  (u4:User {name: 'U4'}),
  (u5:User {name: 'U5'}),

  (d1:Device {id: 'D1'}),
  (d2:Device {id: 'D2'}),
  (d3:Device {id: 'D3'}),

  (u1)-[:USES]->(d1),
  (u2)-[:USES]->(d1),
  (u3)-[:USES]->(d2),
  (u4)-[:USES]->(d2),
  (u5)-[:USES]->(d3),
  (u1)-[:USES]->(d2)
"""

qexec(CREATE_DATA)
print("Dane załadowane")

Brak wyników.
Brak wyników.
Dane załadowane


---
## Polecenie 1a
### Znajdź wszystkich użytkowników, którzy korzystają z tego samego urządzenia co użytkownik U1

**Zapytanie Cypher:**
```cypher
MATCH (u1:User {name: 'U1'})-[:USES]->(d:Device)<-[:USES]-(other:User)
WHERE other <> u1
RETURN DISTINCT other.name AS user, d.id AS shared_device
ORDER BY shared_device, user
```

**Logika:**
1. Zaczynamy od węzła U1
2. Przechodzimy relacją `USES` do urządzenia `d`
3. Cofamy się relacją `USES` do innego użytkownika
4. Wykluczamy samego U1 warunkiem `WHERE other <> u1`

In [12]:
query_1a = """
MATCH (u1:User {name: 'U1'})-[:USES]->(d:Device)<-[:USES]-(other:User)
WHERE other <> u1
RETURN DISTINCT other.name AS user, d.id AS shared_device
ORDER BY shared_device, user
"""

results_1a = qexec(query_1a)

print("Użytkownicy korzystający z tego samego urządzenia co U1:")
print("-" * 40)
for _, row in results_1a.iterrows():
    print(f"  Użytkownik: {row['user']}  |  Wspólne urządzenie: {row['shared_device']}")

# Oczekiwany wynik:
# U2 -> D1 (obaj używają D1)
# U3 -> D2 (obaj używają D2)
# U4 -> D2 (obaj używają D2)

Użytkownicy korzystający z tego samego urządzenia co U1:
----------------------------------------
  Użytkownik: U2  |  Wspólne urządzenie: D1
  Użytkownik: U3  |  Wspólne urządzenie: D2
  Użytkownik: U4  |  Wspólne urządzenie: D2


---
## Polecenie 1b
### Znajdź pary użytkowników korzystających z tego samego urządzenia

**Zapytanie Cypher:**
```cypher
MATCH (a:User)-[:USES]->(d:Device)<-[:USES]-(b:User)
WHERE a.name < b.name
RETURN a.name AS user_a, b.name AS user_b, d.id AS shared_device
ORDER BY shared_device, user_a
```

**Logika:**
1. Dopasowujemy dwóch różnych użytkowników (`a` i `b`) połączonych z tym samym urządzeniem `d`
2. Warunek `a.name < b.name` eliminuje duplikaty — zapobiega zwracaniu pary (A,B) i (B,A) jednocześnie

In [13]:
query_1b = """
MATCH (a:User)-[:USES]->(d:Device)<-[:USES]-(b:User)
WHERE a.name < b.name
RETURN a.name AS user_a, b.name AS user_b, d.id AS shared_device
ORDER BY shared_device, user_a
"""

results_1b = qexec(query_1b)

print("Pary użytkowników korzystających z tego samego urządzenia:")
print("-" * 50)
for _, row in results_1b.iterrows():
    print(f"  ({row['user_a']}, {row['user_b']})  |  Wspólne urządzenie: {row['shared_device']}")

# Oczekiwany wynik:
# (U1, U2) -> D1
# (U1, U3) -> D2
# (U1, U4) -> D2
# (U3, U4) -> D2

Pary użytkowników korzystających z tego samego urządzenia:
--------------------------------------------------
  (U1, U2)  |  Wspólne urządzenie: D1
  (U1, U3)  |  Wspólne urządzenie: D2
  (U1, U4)  |  Wspólne urządzenie: D2
  (U3, U4)  |  Wspólne urządzenie: D2


---
## Polecenie 2 — Dlaczego graf jest dobrym modelem do tego typu zapytań?

Graf naturalnie reprezentuje relacje między obiektami — węzły to obiekty, krawędzie to powiązania. Zapytania takie jak "kto używa tego samego urządzenia" sprowadzają się do prostego przejścia po krawędziach (traversal), bez potrzeby pisania JOIN-ów jak w SQL. Cypher bezpośrednio odzwierciedla intuicję: idź od użytkownika po relacji USES do urządzenia, wróć po tej samej relacji do innych użytkowników.

In [14]:
driver.close()
print("Połączenie zamknięte")

Połączenie zamknięte
